# 03 · Flujo de modelamiento: escalar, imputar y evaluar

**Qué hace este cuaderno.** Oculta valores conocidos de los dos tramos, los rellena con cuatro métodos (media, mediana, moda y KNN) y mide qué tan bien los recupera cada uno.

**Regla central (sin fuga de datos).** Todo lo que aprende de los datos (el escalador, las medias, las medianas, las modas y los vecinos) se ajusta **solo con 2018**. El tramo de 2019 solo se transforma. Los valores ocultos se vuelven vacíos antes de imputar y el error se mide solo en esas celdas.

## 0. Configuración

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Rutas (este cuaderno vive en notebooks/)
TRAMOS = Path("../data/gold/tramos")
MASCARAS = Path("../data/gold/mascaras")

# Semillas: las máscaras v1 (2018) sirven para ajustar y las v2 (2019) para evaluar
SEMILLAS_AJUSTE = range(1, 11)
SEMILLAS_EVAL = range(101, 111)

KS = [2, 3, 5, 7, 9, 11, 13, 15, 19]            # valores de k que se prueban
LINEAS_BASE = ["media", "mediana", "moda"]      # métodos de referencia
MODELO = "KNN (k=9)"                            # modelo que se compara con las líneas base

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

## 1. Datos: los dos tramos

- **Ajuste (2018):** con este tramo se aprende todo y se escogen los parámetros (por ejemplo, $k$).
- **Evaluación (2019):** se usa una sola vez al final, solo para medir.

In [2]:
leer = lambda f: pd.read_csv(TRAMOS / f, index_col="Time_dt", parse_dates=["Time_dt", "Time_original"])

ajuste = leer("tramo_ajuste_2018.csv")
evaluacion = leer("tramo_evaluacion_2019.csv")
variables = [c for c in ajuste.columns if c != "Time_original"]

X_aj = ajuste[variables].to_numpy()          # 2018: ajuste
X_ev = evaluacion[variables].to_numpy()      # 2019: evaluación

print("Ajuste 2018:", X_aj.shape, "| Evaluación 2019:", X_ev.shape, "| Variables:", len(variables))

Ajuste 2018: (1933, 12) | Evaluación 2019: (2415, 12) | Variables: 12


### Diagnóstico: ¿cambió el nivel de las variables entre 2018 y 2019?

Se escalan los dos tramos con la regla de 2018 (promedio y desviación de 2018, sin recalcular para 2019). Si un tramo se parece al otro, el promedio de 2019 debería quedar cerca de 0. Cuando queda lejos, esa variable cambió de nivel entre años.

In [3]:
esc_diag = StandardScaler().fit(X_aj)        # aprende el promedio y la desviación de cada variable SOLO con 2018
X_aj_esc = esc_diag.transform(X_aj)          # 2018 con su propia regla
X_ev_esc = esc_diag.transform(X_ev)          # 2019 con la regla de 2018 (no se recalcula)

print(pd.DataFrame({
    "media 2018": np.nanmean(X_aj_esc, axis=0),
    "desv 2018":  np.nanstd(X_aj_esc, axis=0),
    "media 2019": np.nanmean(X_ev_esc, axis=0),
}, index=variables).round(2))

                         media 2018  desv 2018  media 2019
Indoor Temperature(°C)         -0.0        1.0        0.09
Indoor Humidity(%)             -0.0        1.0        0.77
Outdoor Temperature(°C)         0.0        1.0       -0.03
Outdoor Humidity(%)            -0.0        1.0        0.42
Relative Pressure(hpa)         -0.0        1.0        2.83
Absolute Pressure(hpa)          0.0        1.0        0.93
Wind Speed(km/h)               -0.0        1.0       -0.20
Gust(km/h)                     -0.0        1.0       -0.22
DewPoint(°C)                    0.0        1.0        1.07
WindChill(°C)                   0.0        1.0       -0.02
Hour Rainfall(mm)               0.0        1.0        0.06
Light(lux)                      0.0        1.0       -0.07


> La presión relativa de 2019 queda unas 2,8 desviaciones por encima de la de 2018. Esto pesa mucho al evaluar (sección 7).

## 2. Máscaras de datos faltantes

Cada máscara dice qué celdas se ocultan (`True` = oculto). Se decidieron una sola vez, para que todos los métodos reciban exactamente los mismos huecos.

In [4]:
def cargar_mascaras(nombre_npz, nombre_indice):
    d = np.load(MASCARAS / nombre_npz, allow_pickle=True)
    assert list(d["columnas"]) == variables           # mismas variables que los tramos
    return d["mascaras"], pd.read_csv(MASCARAS / nombre_indice)   # (escenarios, filas, variables) + índice

masc_aj, indice_aj = cargar_mascaras("mascaras_ajuste_v1.npz", "indice_ajuste_v1.csv")
masc_ev, indice_ev = cargar_mascaras("mascaras_evaluacion_v2.npz", "indice_evaluacion_v2.csv")
assert masc_aj.shape[1:] == X_aj.shape and masc_ev.shape[1:] == X_ev.shape

print("Escenarios de ajuste (v1):", len(indice_aj), "| de evaluación (v2):", len(indice_ev))

Escenarios de ajuste (v1): 1140 | de evaluación (v2): 1220


### Ejemplo de un escenario

In [5]:
e = indice_aj[(indice_aj.patron == "overlap") & (indice_aj.q == 3) & (indice_aj.duracion == "1d")
              & (indice_aj.pct_objetivo == 20) & (indice_aj.semilla == 1)].iloc[0]
mascara = masc_aj[e["id"]]                   # True = valor oculto
print(e[["patron", "q", "duracion", "pct_objetivo", "pct_real", "sensores"]].to_dict())

X_oculto = X_aj.copy()
X_oculto[mascara] = np.nan                   # lo oculto pasa a ser vacío; X_aj queda intacto
print("Celdas ocultas:", mascara.sum(), "| % del tramo:", round(mascara.mean() * 100, 1))

{'patron': 'overlap', 'q': 3, 'duracion': '1d', 'pct_objetivo': 20, 'pct_real': 19.66, 'sensores': 'interior|pluviometro|anemometro'}
Celdas ocultas: 4560 | % del tramo: 19.7


## 3. El flujo (`Pipeline`)

Cada método es un `Pipeline` de `scikit-learn`. Los pasos van en este orden:

| Paso | Qué hace | Por qué va en este lugar |
|---|---|---|
| `escalar` | `MinMaxScaler`: pone cada variable entre 0 y 1 | KNN compara distancias. Sin esto, la luz (miles de lux) pesaría más que la temperatura |
| `imputar` | `KNNImputer` (o `SimpleImputer` con media, mediana o moda) | Va después de escalar, para que los vecinos se busquen con todas las variables en la misma escala |
| inverso | `inverse_transform` del escalador | Devuelve lo imputado a grados, hPa y lux, para medir el error en unidades reales |

**Cómo se evita la fuga de datos**

- `fit` se hace solo con 2018. El tramo de 2019 solo pasa por `transform`, con las reglas de 2018.
- `clone` crea una copia sin ajustar para cada escenario, así ninguno hereda lo que aprendió otro.
- Los valores ocultos se vuelven vacíos **antes** de imputar, y el error se mide solo en esas celdas.

**Convención para agregar métodos (FKNN, regresión lineal):** el escalador se llama `"escalar"` y el imputador `"imputar"`. Así el código sabe cuándo devolver los valores a sus unidades originales.

In [6]:
METODOS = {
    "media":     Pipeline([("imputar", SimpleImputer(strategy="mean", keep_empty_features=True))]),
    "mediana":   Pipeline([("imputar", SimpleImputer(strategy="median", keep_empty_features=True))]),
    "moda":      Pipeline([("imputar", SimpleImputer(strategy="most_frequent", keep_empty_features=True))]),
    "KNN (k=9)": Pipeline([("escalar", MinMaxScaler()),
                           ("imputar", KNNImputer(n_neighbors=9, weights="uniform", keep_empty_features=True))]),
    # "FKNN":              Pipeline([...]),   # lo agrega el compañero
    # "Regresión lineal":  Pipeline([...]),   # lo agrega el compañero
}

## 4. Funciones de evaluación

- `evaluar`: devuelve el $R^2$ (promedio de las variables afectadas) y el MSE en escala 0 a 1, **solo sobre las celdas ocultas**.
- `correr`: corre los métodos sobre un mismo tramo (se usa en 2018 para ajustar).
- `correr_ev`: ajusta con 2018 y evalúa en 2019 (evaluación final).
- `metricas`: MSE y $R^2$ por variable, para un solo escenario.

In [7]:
RANGO = np.nanmax(X_aj, axis=0) - np.nanmin(X_aj, axis=0)   # solo para reportar el MSE en escala 0 a 1

def evaluar(X_real, mascara, X_imp):
    """R² (promedio de las variables afectadas) y MSE en escala 0-1, solo sobre celdas ocultas."""
    celdas = mascara & ~np.isnan(X_real)
    cols = np.where(celdas.any(axis=0))[0]
    r2 = [r2_score(X_real[celdas[:, j], j], X_imp[celdas[:, j], j]) for j in cols]
    err = ((X_imp - X_real) / RANGO)[celdas]
    return np.mean(r2), np.mean(err ** 2)


def correr(metodos, indice, masc, X, semillas):
    """Ajuste en 2018: cada método aprende solo con los valores visibles del mismo tramo."""
    filas = []
    for e in indice[indice["semilla"].isin(semillas)].itertuples():
        X_oculto = X.copy()
        X_oculto[masc[e.id]] = np.nan
        for nombre, flujo in metodos.items():
            f = clone(flujo)
            Z = f.fit_transform(X_oculto)                       # aprende solo con lo visible
            if "escalar" in f.named_steps:
                Z = f.named_steps["escalar"].inverse_transform(Z)
            r2, mse = evaluar(X, masc[e.id], Z)
            filas.append(dict(metodo=nombre, patron=e.patron, q=e.q, duracion=e.duracion,
                              pct=e.pct_objetivo, semilla=e.semilla, R2=r2, MSE=mse))
    return pd.DataFrame(filas)


def metricas(X_real, mascara, X_imp, variables):
    """MSE y R² por variable, calculados solo sobre las celdas ocultas."""
    celdas = mascara & ~np.isnan(X_real)     # celdas ocultas cuyo valor real sí conocemos
    filas = []
    for j, v in enumerate(variables):
        if celdas[:, j].any():
            real, pred = X_real[celdas[:, j], j], X_imp[celdas[:, j], j]
            filas.append(dict(variable=v,
                              MSE=mean_squared_error(real, pred),
                              R2=r2_score(real, pred)))
    return pd.DataFrame(filas).set_index("variable")

In [8]:
def correr_ev(metodos, indice, masc, X_train, X_test, semillas, variables=variables):
    """Evaluación final: ajusta TODO el pipeline con 2018 (X_train) e imputa 2019 (X_test)."""
    glob, porvar = [], []

    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)

    # El rango se calcula exclusivamente con 2018
    rango = np.nanmax(X_train, axis=0) - np.nanmin(X_train, axis=0)
    rango[~np.isfinite(rango) | (rango == 0)] = 1.0

    seleccion = indice[indice["semilla"].isin(list(semillas))]

    if seleccion.empty:
        raise ValueError(
            f"No hay escenarios para las semillas solicitadas. "
            f"Disponibles: {sorted(indice['semilla'].unique())}"
        )

    for e in seleccion.itertuples(index=False):
        m = np.asarray(masc[e.id], dtype=bool)

        X_oculto = X_test.copy()
        X_oculto[m] = np.nan

        # Solo se evalúan las celdas ocultas con valor real conocido
        celdas = m & np.isfinite(X_test)
        cols = np.where(celdas.any(axis=0))[0]

        base = {
            "id": e.id,
            "patron": e.patron,
            "q": e.q,
            "duracion": e.duracion,
            "pct": e.pct_objetivo,
            "semilla": e.semilla,
        }

        for nombre, flujo in metodos.items():

            # Ajustar TODO el pipeline exclusivamente con 2018
            f = clone(flujo)
            f.fit(X_train)

            # Imputar 2019 utilizando el modelo ajustado con 2018
            Z = f.transform(X_oculto)
            Z = np.asarray(Z, dtype=float)

            # Volver a las unidades originales si hay escalador
            if hasattr(f, "named_steps") and "escalar" in f.named_steps:
                Z = f.named_steps["escalar"].inverse_transform(Z)

            # R²: promedio de las variables evaluables
            r2_columnas = []
            for j in cols:
                valido = celdas[:, j] & np.isfinite(Z[:, j])

                if valido.sum() >= 2:
                    r2_columnas.append(
                        r2_score(X_test[valido, j], Z[valido, j])
                    )

            r2 = np.mean(r2_columnas) if r2_columnas else np.nan

            # MSE normalizado con el rango de 2018
            validas = celdas & np.isfinite(Z)
            if validas.any():
                mse = np.mean(
                    (((Z - X_test) / rango)[validas]) ** 2
                )
            else:
                mse = np.nan

            glob.append({
                **base,
                "metodo": nombre,
                "R2": r2,
                "MSE": mse,
            })

            # Error por variable
            for j in cols:
                valido = celdas[:, j] & np.isfinite(Z[:, j])
                err2 = (
                    Z[valido, j] - X_test[valido, j]
                ) ** 2

                porvar.append({
                    **base,
                    "metodo": nombre,
                    "variable": variables[j],
                    "sse": err2.sum(),
                    "n": len(err2),
                })

    return pd.DataFrame(glob), pd.DataFrame(porvar)

## 5. Verificación de no fuga de datos

### 5.1 El escalador aprende solo con lo visible

Con la máscara de ejemplo, el escalador se ajusta con los valores que se ven. Debe dar casi lo mismo que con todo el tramo, salvo en las variables de los equipos que fallan.

In [9]:
esc_oculto = StandardScaler().fit(X_oculto)  # aprende SOLO con lo visible (los vacíos se ignoran)
esc_completo = StandardScaler().fit(X_aj)    # solo para comparar; este NO se usa en el flujo

X_esc = esc_oculto.transform(X_oculto)
print("Siguen vacías las celdas ocultas:", np.isnan(X_esc[mascara]).all())

print(pd.DataFrame({
    "media (sin ocultas)": esc_oculto.mean_,
    "media (completo)":    esc_completo.mean_,
    "desv (sin ocultas)":  esc_oculto.scale_,
    "desv (completo)":     esc_completo.scale_,
}, index=variables).round(2))

Siguen vacías las celdas ocultas: True
                         media (sin ocultas)  media (completo)  desv (sin ocultas)  desv (completo)
Indoor Temperature(°C)                 23.04             23.05                1.61             1.68
Indoor Humidity(%)                     64.60             64.35                5.17             5.44
Outdoor Temperature(°C)                19.55             19.55                3.34             3.34
Outdoor Humidity(%)                    77.53             77.53               14.02            14.02
Relative Pressure(hpa)               1010.87           1010.87                1.10             1.10
Absolute Pressure(hpa)                825.57            825.57                1.10             1.10
Wind Speed(km/h)                        2.93              2.96                2.62             2.69
Gust(km/h)                              4.96              5.00                3.51             3.57
DewPoint(°C)                           15.22             15.2

> Solo cambian un poco las variables de los equipos que fallaron en esta máscara. Las demás salen idénticas.

### 5.2 Comprobaciones automáticas del flujo

Si alguna falla, hay fuga de datos.

In [10]:
# 1) El escalador de KNN solo conoce 2018
f = clone(METODOS[MODELO]).fit(X_aj)
esc = f.named_steps["escalar"]
min_antes, max_antes = esc.data_min_.copy(), esc.data_max_.copy()
assert np.allclose(min_antes, np.nanmin(X_aj, axis=0)) and np.allclose(max_antes, np.nanmax(X_aj, axis=0))

# 2) Pasar 2019 por transform no cambia lo aprendido (no se recalcula)
f.transform(X_ev)
assert np.allclose(min_antes, esc.data_min_) and np.allclose(max_antes, esc.data_max_)

# 3) La media de la línea base es la de 2018
b = clone(METODOS["media"]).fit(X_aj)
assert np.allclose(b.named_steps["imputar"].statistics_, np.nanmean(X_aj, axis=0))

print("Comprobaciones superadas: el flujo solo aprende con 2018.")

Comprobaciones superadas: el flujo solo aprende con 2018.


### 5.3 Las líneas base en el escenario de ejemplo, por variable

In [11]:
bases = {k: METODOS[k] for k in ("media", "mediana")}

tablas = {}
for nombre, flujo in bases.items():
    X_imp = clone(flujo).fit_transform(X_oculto)
    tablas[nombre] = metricas(X_aj, mascara, X_imp, variables)
    print("\n", nombre)
    print(tablas[nombre].round(3))
    print("R² promedio:", round(tablas[nombre]["R2"].mean(), 3))


 media
                           MSE     R2
variable                             
Indoor Temperature(°C)   3.087 -0.000
Indoor Humidity(%)      33.001 -0.009
Wind Speed(km/h)         7.623 -0.000
Gust(km/h)              13.308 -0.001
Hour Rainfall(mm)        0.243 -0.000
R² promedio: -0.002

 mediana
                           MSE     R2
variable                             
Indoor Temperature(°C)   3.310 -0.073
Indoor Humidity(%)      36.459 -0.114
Wind Speed(km/h)         7.864 -0.032
Gust(km/h)              15.393 -0.157
Hour Rainfall(mm)        0.255 -0.051
R² promedio: -0.086


> Con media y mediana el $R^2$ queda cerca de 0 o por debajo: son lo mínimo que un modelo debe superar.

## 6. Ajuste en 2018 (máscaras v1)

Aquí se escogen los parámetros. El tramo de 2019 no se toca.

### 6.1 Líneas base y KNN en 2018

Métricas principales (las de la primera entrega): MSE y $R^2$.

In [12]:
res = correr(METODOS, indice_aj, masc_aj, X_aj, semillas=SEMILLAS_AJUSTE)

orden = list(METODOS)
print(res.groupby(["patron", "metodo"])[["R2", "MSE"]].mean().round(3).unstack(0))
print()
print(res.pivot_table(index=["patron", "pct"], columns="metodo", values="R2")[orden].round(3))

                R2              MSE        
patron    disjoint overlap disjoint overlap
metodo                                     
KNN (k=9)    0.349   0.204    0.018   0.023
media       -0.037  -0.036    0.034   0.033
mediana     -0.101  -0.114    0.036   0.036
moda        -0.360  -0.354    0.047   0.045

metodo        media  mediana   moda  KNN (k=9)
patron   pct                                  
disjoint 5   -0.112   -0.147 -0.336      0.377
         10  -0.034   -0.098 -0.328      0.393
         20  -0.011   -0.086 -0.364      0.455
         30  -0.018   -0.088 -0.379      0.346
         40  -0.013   -0.087 -0.393      0.216
         50  -0.013   -0.086 -0.381      0.195
overlap  5   -0.080   -0.160 -0.382      0.162
         10  -0.066   -0.132 -0.348      0.184
         20  -0.020   -0.104 -0.301      0.241
         30  -0.021   -0.098 -0.314      0.195
         40  -0.015   -0.099 -0.391      0.225
         50  -0.013   -0.089 -0.396      0.222


### 6.2 Número de vecinos ($k$)

In [13]:
metodos_k = {
    f"k={k}": Pipeline([("escalar", MinMaxScaler()),
                        ("imputar", KNNImputer(n_neighbors=k, weights="uniform", keep_empty_features=True))])
    for k in KS
}

res_k = correr(metodos_k, indice_aj, masc_aj, X_aj, semillas=[1, 2, 3])   # para la corrida final: SEMILLAS_AJUSTE

orden_k = list(metodos_k)
print("R² promedio (más alto es mejor)")
print(res_k.pivot_table(index="patron", columns="metodo", values="R2")[orden_k].round(3))
print("\nMSE promedio en escala 0-1 (más bajo es mejor)")
print(res_k.pivot_table(index="patron", columns="metodo", values="MSE")[orden_k].round(4))

R² promedio (más alto es mejor)
metodo      k=2    k=3    k=5    k=7    k=9   k=11  k=13   k=15   k=19
patron                                                                
disjoint  0.115  0.207  0.283  0.311  0.333  0.350  0.36  0.365  0.375
overlap   0.074  0.132  0.173  0.191  0.201  0.208  0.21  0.212  0.215

MSE promedio en escala 0-1 (más bajo es mejor)
metodo       k=2     k=3     k=5     k=7     k=9    k=11    k=13    k=15    k=19
patron                                                                          
disjoint  0.0241  0.0218  0.0198  0.0188  0.0182  0.0179  0.0176  0.0175  0.0173
overlap   0.0267  0.0253  0.0242  0.0238  0.0235  0.0234  0.0234  0.0233  0.0233


**Decisión: $k=9$.** El artículo advierte que, a mayor $k$, puede aumentar el sesgo. Necesitamos un $k$ que no sea muy alto pero que funcione bien. A partir de $k=9$ el modelo cambia poco, así que nos quedamos con 9.

### 6.3 Pesos de los vecinos: ¿importan?

In [14]:
metodos_w = {
    "uniforme":  Pipeline([("escalar", MinMaxScaler()),
                           ("imputar", KNNImputer(n_neighbors=9, weights="uniform", keep_empty_features=True))]),
    "distancia": Pipeline([("escalar", MinMaxScaler()),
                           ("imputar", KNNImputer(n_neighbors=9, weights="distance", keep_empty_features=True))]),
}

res_w = correr(metodos_w, indice_aj, masc_aj, X_aj, semillas=[1, 2, 3])

print(res_w.groupby(["patron", "metodo"])[["R2", "MSE"]].mean().round(4).unstack(1))

# escenario por escenario: ¿en cuántos gana cada uno?
dif = res_w.pivot_table(index=["patron", "q", "duracion", "pct", "semilla"], columns="metodo", values="R2")
dif["d"] = dif["distancia"] - dif["uniforme"]
print("\nDiferencia de R² (distancia - uniforme): promedio", round(dif["d"].mean(), 4),
      "| mediana", round(dif["d"].median(), 4),
      "| % de escenarios donde gana distancia:", round((dif["d"] > 0).mean() * 100, 1))

print()
print(res_w.pivot_table(index=["patron", "pct"], columns="metodo", values="R2").round(3))

                R2                MSE         
metodo   distancia uniforme distancia uniforme
patron                                        
disjoint    0.2764   0.3325    0.0199   0.0182
overlap     0.1750   0.2015    0.0243   0.0235

Diferencia de R² (distancia - uniforme): promedio -0.0436 | mediana -0.0022 | % de escenarios donde gana distancia: 28.9

metodo        distancia  uniforme
patron   pct                     
disjoint 5        0.271     0.297
         10       0.315     0.372
         20       0.373     0.457
         30       0.300     0.354
         40       0.147     0.212
         50       0.154     0.210
overlap  5        0.133     0.162
         10       0.167     0.205
         20       0.250     0.254
         30       0.131     0.173
         40       0.200     0.220
         50       0.175     0.198


> Los pesos iguales (`uniform`) funcionaron mejor que los pesos por distancia, así que se mantienen.

## 7. Evaluación final en 2019 (máscaras v2)

Todo el flujo se ajusta con 2018 y se aplica a 2019, que se usa una sola vez.

In [15]:
res_ev, detalle_ev = correr_ev(
    metodos=METODOS,
    indice=indice_ev,
    masc=masc_ev,
    X_train=X_aj,       # 2018: entrenamiento
    X_test=X_ev,        # 2019: evaluación
    semillas=SEMILLAS_EVAL
)

print("Dimensiones:", res_ev.shape)
print("Columnas:", res_ev.columns.tolist())
display(res_ev.head())

Dimensiones: (4880, 9)
Columnas: ['id', 'patron', 'q', 'duracion', 'pct', 'semilla', 'metodo', 'R2', 'MSE']


,id,patron,q,duracion,pct,semilla,metodo,R2,MSE
0,0,overlap,3,1h,5,101,media,-0.255628,0.031179
1,0,overlap,3,1h,5,101,mediana,-0.229497,0.030581
2,0,overlap,3,1h,5,101,moda,-0.454224,0.038298
3,0,overlap,3,1h,5,101,KNN (k=9),-0.261772,0.028598
4,1,overlap,3,1h,5,102,media,-1.346563,0.077615


### 7.1 Resultados por patrón y comparación con las líneas base

In [16]:
orden = list(METODOS)
print(res_ev.groupby(["patron", "metodo"])[["R2", "MSE"]].mean().round(3).unstack(0))
print()
print(res_ev.pivot_table(index=["patron", "pct"], columns="metodo", values="R2")[orden].round(3))

# ¿en cuántos escenarios KNN le gana a la mejor línea base?
mejor_base = (res_ev[res_ev.metodo.isin(LINEAS_BASE)]
              .groupby(["patron", "q", "duracion", "pct", "semilla", "metodo"])["MSE"].mean()
              .unstack().min(axis=1))
knn = res_ev[res_ev.metodo == MODELO].set_index(["patron", "q", "duracion", "pct", "semilla"])["MSE"]
gana = knn < mejor_base.reindex(knn.index)
print("\nKNN tiene menor MSE que la mejor línea base en", round(gana.mean() * 100, 1), "% de los escenarios")
print(gana.groupby(level="patron").mean().round(3))

                R2              MSE        
patron    disjoint overlap disjoint overlap
metodo                                     
KNN (k=9)   -0.620  -0.949    0.042   0.051
media       -1.135  -1.183    0.056   0.057
mediana     -1.110  -1.155    0.056   0.057
moda        -1.047  -1.050    0.057   0.056

metodo        media  mediana   moda  KNN (k=9)
patron   pct                                  
disjoint 5   -1.408   -1.392 -1.191     -0.826
         10  -1.420   -1.369 -1.201     -0.888
         20  -0.827   -0.807 -0.889     -0.196
         30  -0.958   -0.939 -0.954     -0.426
         40  -1.155   -1.135 -1.066     -0.790
         50  -0.863   -0.850 -0.875     -0.551
overlap  5   -1.514   -1.452 -1.214     -1.209
         10  -1.304   -1.261 -1.092     -1.084
         20  -1.082   -1.070 -0.998     -0.838
         30  -1.110   -1.096 -1.007     -0.913
         40  -1.090   -1.072 -1.010     -0.880
         50  -0.944   -0.932 -0.955     -0.711

KNN tiene menor MSE que la mejor

### 7.2 ¿En qué variables falla más?

El error de cada variable se divide entre su varianza (NMSE). Un valor de 1 equivale a adivinar el promedio, y $R^2 = 1 - \text{NMSE}$.

In [17]:
var_ev = pd.Series(
    np.nanvar(X_ev, axis=0),
    index=variables
)

# Usar detalle_ev, que devuelve correr_ev
pv = detalle_ev.groupby(
    ["patron", "metodo", "variable"]
)[["sse", "n"]].sum()

# NMSE = MSE por variable / varianza de esa variable
varianzas = pv.index.get_level_values("variable").map(var_ev)

pv["NMSE"] = np.where(
    (pv["n"].to_numpy() > 0) & (varianzas.to_numpy() > 0),
    pv["sse"].to_numpy() / pv["n"].to_numpy() / varianzas.to_numpy(),
    np.nan
)


tabla = pv["NMSE"].unstack("metodo")[["media", MODELO]]
tabla["KNN / media"] = tabla[MODELO] / tabla["media"]
tabla["R2 KNN"] = 1 - tabla[MODELO]
for patron in ["disjoint", "overlap"]:
    print("\n", patron)
    print(tabla.loc[patron].sort_values(MODELO, ascending=False).round(2))

knn_r = res_ev[res_ev.metodo == MODELO]
print()
print(knn_r.pivot_table(index="duracion", columns="patron", values="R2").reindex(["1h", "6h", "1d", "3d", "7d"]).round(3))
print(knn_r.pivot_table(index="q", columns="patron", values="R2").round(3))


 disjoint
metodo                   media  KNN (k=9)  KNN / media  R2 KNN
variable                                                      
Relative Pressure(hpa)   10.46      10.72         1.02   -9.72
Absolute Pressure(hpa)    2.04       2.15         1.06   -1.15
DewPoint(°C)              2.38       1.31         0.55   -0.31
Hour Rainfall(mm)         1.01       1.13         1.12   -0.13
Wind Speed(km/h)          1.05       0.80         0.76    0.20
Gust(km/h)                1.06       0.79         0.74    0.21
Indoor Temperature(°C)    0.98       0.74         0.75    0.26
Indoor Humidity(%)        1.62       0.73         0.45    0.27
Light(lux)                1.00       0.70         0.70    0.30
Outdoor Humidity(%)       1.22       0.44         0.36    0.56
WindChill(°C)             0.99       0.35         0.36    0.65
Outdoor Temperature(°C)   0.99       0.34         0.35    0.66

 overlap
metodo                   media  KNN (k=9)  KNN / media  R2 KNN
variable                          

### 7.3 Variabilidad entre escenarios

In [18]:
tabla = res_ev.groupby(["patron", "pct", "metodo"])["R2"].agg(["mean", "std"]).round(3)
print(tabla)

                         mean    std
patron   pct metodo                 
disjoint 5   KNN (k=9) -0.826  1.527
             media     -1.408  1.531
             mediana   -1.392  1.437
             moda      -1.191  1.081
         10  KNN (k=9) -0.888  1.805
             media     -1.420  1.515
             mediana   -1.369  1.480
             moda      -1.201  1.117
         20  KNN (k=9) -0.196  0.885
             media     -0.827  0.709
             mediana   -0.807  0.703
             moda      -0.889  0.477
         30  KNN (k=9) -0.426  0.867
             media     -0.958  0.672
             mediana   -0.939  0.678
             moda      -0.954  0.454
         40  KNN (k=9) -0.790  0.595
             media     -1.155  0.546
             mediana   -1.135  0.539
             moda      -1.066  0.397
         50  KNN (k=9) -0.551  0.577
             media     -0.863  0.519
             mediana   -0.850  0.513
             moda      -0.875  0.378
overlap  5   KNN (k=9) -1.209  1.052
 